# 2. Prompt Management

**Goal:** Create, version, and programmatically retrieve a DataRobot Prompt Template so your agent can load a consistent system prompt at runtime.

**Key Concept:**
Prompts are managed, versioned assets in DataRobot (not hardcoded strings). This notebook fetches a Prompt Template by `PROMPT_TEMPLATE_ID`, optionally selects a version, renders it with variables, and injects the rendered result as the agent’s `system_prompt`.

## What you’ll do

1. Create a Prompt Template in the DataRobot UI.
2. Store the template ID in `.env` as `PROMPT_TEMPLATE_ID`.
3. Fetch the template and choose a version (`v1`, `v2`, or latest).
4. Render the prompt with variables (e.g., `company_name`).
5. Use the rendered text as the agent’s `system_prompt` for a quick runtime validation.


### Create the Prompt Template in DataRobot (required)

This notebook expects an existing **Prompt Template** in DataRobot. If you see an error like `PromptTemplate ... not found`, it means the `PROMPT_TEMPLATE_ID` in your `.env` file does not point to a template in *your* tenant.

#### Steps (UI)

1. In DataRobot, go to **Registry => Prompts** (Prompt Management).
2. Click **Create Prompt**.
3. Create a **System Prompt** template (recommended) and paste the example below.
4. Save the template.
5. Copy the **Prompt Template ID** from the template URL and add it to your `.env` file as `PROMPT_TEMPLATE_ID=<your-prompt-template-id>`.

#### Example prompt settings (paste into the template and update IDs before saving)

Name: `System Prompt - Notebook 2`

Description: `Used for Agent Build Clinic`

Prompt text: 

```
You are a helpful forecasting assistant working for {{ company_name }}.

- Use the forecasting deployment with ID `<your-forecast-deployment-id>`.
- Forecast using the scoring dataset with ID `<your-scoring-dataset-id>`.
```

Variable name: `company_name` - will pop up

Description: `Company name`


In [ ]:
# 1. Imports & setup
import os
from dotenv import load_dotenv
import datarobot as dr
from datarobot.models.genai.prompt_template import PromptTemplate
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

# 2. Load configuration
load_dotenv(override=True)

# 3. Initialize DataRobot client
dr_client = dr.Client()
print(f"Connected to DataRobot: {dr_client.endpoint}")

# 4. Read configuration from environment
PROMPT_TEMPLATE_ID = os.getenv("PROMPT_TEMPLATE_ID")
# Version to test (set to "v1", "v2", or None for latest)
PROMPT_VERSION_ID = "v1"

RENDER_VARIABLES = {
    "company_name": "DataRobot Forecasting Inc.",
}

MODEL_NAME = os.getenv("MODEL_NAME")

if not PROMPT_TEMPLATE_ID:
    raise ValueError(
        "PROMPT_TEMPLATE_ID is not set.\n"
        "Fix: add `PROMPT_TEMPLATE_ID=<your-prompt-template-id>` to your `.env` in the repo root (see README)."
    )

# 5. Fetch Prompt Template + select a version
print(f"\n--- Fetching Template: {PROMPT_TEMPLATE_ID} ---")
template = PromptTemplate.get(PROMPT_TEMPLATE_ID)
print(f"Template Name: {template.name}")

target_version = None
if PROMPT_VERSION_ID:
    # Logic to handle "v1", "V1", or 1
    versions = template.list_versions()
    search_str = str(PROMPT_VERSION_ID).lower().replace("v", "")

    for v in versions:
        # Check exact ID match OR version number match
        if v.id == PROMPT_VERSION_ID:
            target_version = v
            break
        if hasattr(v, "version") and str(v.version) == search_str:
            target_version = v
            break

    if not target_version:
        available = [f"v{getattr(v, 'version', '?')} (ID: {v.id})" for v in versions]
        raise ValueError(
            f"Could not find version '{PROMPT_VERSION_ID}'.\nAvailable: {available}"
        )
else:
    print("Fetching latest version...")
    target_version = template.get_latest_version()

print(
    f"Selected Version: v{getattr(target_version, 'version', '?')} (ID: {target_version.id})"
)

# 6. Render prompt text (this becomes the agent's system_prompt)
try:
    system_prompt = target_version.render(variables=RENDER_VARIABLES)
    print("\n" + "=" * 30)
    print("  RENDERED SYSTEM PROMPT")
    print("=" * 30)
    print(system_prompt)
    print("=" * 30)
except Exception as e:
    print(f"\nCRITICAL ERROR: {e}")
    if hasattr(target_version, "variables"):
        print(f"REQUIRED Variables: {target_version.variables}")
    print(f"PROVIDED Variables: {list(RENDER_VARIABLES.keys())}")
    raise e

# 7. Configure the LLM (via DataRobot LLM Gateway)
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token,
        base_url=dr_client.endpoint + "/genai/llmgw",
    ),
)

# 8. Define the agent using the rendered prompt
agent = Agent(model=model, system_prompt=system_prompt)

# 9. Execution (sanity-check)
print("\n--- Running Test Query ---")
test_query = "Hello, who are you and who do you work for?"
print(f"User: {test_query}\n")

async with agent:
    response = await agent.run(test_query)
    print(response.output)